# Quanvolutional Neural Network (QCNN) for Speech Recognition
Speech recognition and audio processing are rapidly expanding frontiers in Quantum AI. In literature, standard classical networks process speech by extracting a Mel-spectrogram (a visual matrix representation of audio frequencies over time).

A prominent architecture for this is the Quanvolutional Neural Network (QCNN). Instead of using classical mathematical matrix kernels (like a standard CNN) to extract acoustic features, a QCNN uses a localized Parameterized Quantum Circuit (PQC) that sweeps across the spectrogram. By leveraging quantum entanglement, the quantum kernel extracts non-local acoustic features and hidden temporal variations that standard classical filters often miss.

We will build a Hybrid Quantum Spoken Command Classifier. The script takes speech audio (represented as extracted Mel-frequency features), processes it through an active Quantum Convolutional Filter (Quanvolution) via PennyLane, and outputs a feature map ready for a speech classification head.

## The Hybrid Acoustic Architecture


> Audio Waveform ──> [ Mel Spectrogram (Time x Freq) ] ──> [ Sweeping Quantum Kernel (PQC) ]
                                                                       │
                                                                       ▼
[ Spoken Command Text ] <── [ Classical Linear Classifier ] <── [ Extracted Quantum Features ]



In [ ]:
# Install required packages (quiet mode for cleaner output)
!pip install --quiet pennylane
!pip install --quiet amazon-braket-pennylane-plugin

### Step 1: Synthesize Acoustic Feature Arrays
We will simulate a small window of a processed speech command (such as a user saying "Yes" vs. "No"). We extract a localized $4 \times 4$ frame from an audio spectrogram.Python

In [ ]:
import pennylane as qml
from pennylane import numpy as np

# Simulate a 4x4 frame of audio spectrogram features (Frequency bins over Time)
# Command 0 (e.g., "Yes"): Characterized by high-frequency bursts at the end
audio_yes = np.array([
    [0.1, 0.2, 0.1, 0.8],
    [0.0, 0.1, 0.2, 0.9],
    [0.2, 0.1, 0.0, 0.7],
    [0.1, 0.0, 0.1, 0.8]
])

# Command 1 (e.g., "No"): Characterized by a strong low-frequency acoustic baseline
audio_no = np.array([
    [0.7, 0.8, 0.1, 0.0],
    [0.9, 0.6, 0.0, 0.1],
    [0.8, 0.7, 0.2, 0.0],
    [0.7, 0.8, 0.1, 0.1]
])

# Stack into a mini speech batch
speech_dataset = np.array([audio_yes, audio_no])
labels = np.array([0, 1])  # 0: "Yes", 1: "No"

### Step 2: Defining the Sweeping Quanvolutional Filter
We will set up a 4-qubit quantum device. The quantum circuit will function as a $2 \times 2$ scanning kernel window that slides across the audio grid, transforming classical acoustic data into structured quantum feature maps.

In [ ]:
num_wires = 4
dev = qml.device("braket.local.qubit", wires=num_wires)

@qml.qnode(dev)
def quanvolution_kernel(spatial_features, weights):
    """
    Processes a 2x2 localized window of speech features.
    spatial_features: 4 raw acoustic values inside the current sliding window.
    """
    # 1. Encode the 2x2 audio block into the 4 qubits using Angle Embedding
    qml.AngleEmbedding(features=spatial_features, wires=range(num_wires), rotation='X')

    # 2. Entangling Phase (Extracts non-local phase dependencies across frequency/time)
    for i in range(num_wires):
        qml.RY(weights[i], wires=i)
    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[1, 2])
    qml.CNOT(wires=[2, 3])
    qml.CNOT(wires=[3, 0])

    # 3. Measure expectation values to generate 4 filtered acoustic feature maps
    return [qml.expval(qml.PauliZ(i)) for i in range(num_wires)]

### Step 3: Executing the Quantum Feature Extractor Loop
We loop through our raw audio spectrogram data, sweeping the $2 \times 2$ quantum filter across the grid with a stride of 2.

In [ ]:
def extract_quantum_speech_features(audio_matrix, q_weights):
    """Slides the quantum kernel over a 4x4 audio matrix to generate an enhanced map."""
    output_features = []

    # Sweep across spatial dimensions (Stride = 2)
    for r in [0, 2]:
        for c in [0, 2]:
            # Extract the 2x2 sub-window from the spectrogram
            window = audio_matrix[r:r+2, c:c+2].flatten()

            # Run through the Amazon Braket simulator device
            q_features = quanvolution_kernel(window, q_weights)
            output_features.extend(q_features)

    return np.array(output_features)

# Initialize 4 trainable weights for our quantum acoustic filter
np.random.seed(42)
kernel_weights = np.random.uniform(0, 2 * np.pi, num_wires, requires_grad=True)

# Process our speech batch through the quantum loop
processed_speech_features = np.array([extract_quantum_speech_features(img, kernel_weights) for img in speech_dataset])
print(f"Extracted Quantum Feature Vector Shape: {processed_speech_features.shape} (Batch, Features)")

### Step 4: Hybrid Classification and Softmax Training
Now that the quantum device has isolated the primary features, we use a classical single-layer linear model to predict the probability of whether the word spoken was "Yes" or "No".

In [ ]:
# Initialize classical classification head weights (16 inputs -> 2 outputs)
classical_weights = np.random.uniform(-0.5, 0.5, (16, 2), requires_grad=True)
classical_bias = np.zeros(2, requires_grad=True)

def softmax(x):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / np.sum(e_x, axis=-1, keepdims=True)

def hybrid_speech_model(audio_input, q_weights, c_weights, c_bias):
    # Step 1: Run through Quantum Filter
    q_features = extract_quantum_speech_features(audio_input, q_weights)
    # Step 2: Pass to Classical Network
    logits = np.dot(q_features, c_weights) + c_bias
    return softmax(logits)

# A quick demonstration forward pass
test_probs = hybrid_speech_model(audio_yes, kernel_weights, classical_weights, classical_bias)
print(f"\nInitial Softmax Output Probabilities for 'Yes' input: [Class 0 (Yes): {test_probs[0]*100:.1f}%,  Class 1 (No): {test_probs[1]*100:.1f}%]")

## Core Quantum Speech Research Tracks
We want to target standard international speech conferences (like IEEE ICASSP or Interspeech), this foundational architecture opens up three prominent research areas:

**Robust Spoken Command Recognition Against Accents**
Standard automatic speech recognition (ASR) systems struggle when encountering non-native accents because the localized phoneme layouts drift from the expected training sample sets.
*   Research Project: Download the Google Speech Commands Dataset (v0.01/v0.02) which contains thousands of short audio clips. Have your students extract Mel-Frequency Cepstral Coefficients (MFCCs), pass them into this hybrid QCNN pipeline, and measure if the quantum feature maps offer higher structural generalization across speaker variations compared to a classical CNN layer of equivalent scale.

**Quantum Reservoir Computing (QRC) for Continuous Text Output**
For real-time continuous speech processing, a sliding spatial window is often limited because it lacks an inherent timeline memory to chain letters together into sentences.
*   Research Project: Instead of a standard spatial loop, look into Quantum Reservoir Computing. You pass the continuous sound wave values sequentially as a time series into a highly disordered transverse Ising model on Amazon Braket. The passive physical entanglement dynamics of the qubits act as a natural recurrent memory block ("echo state property"). The readout layer is then trained via simple ridge regression to predict words based on the lingering multi-qubit correlations.

**Acoustic Defense Against Adversarial Noise Attacks**
Speech AI systems are highly vulnerable to adversarial noise—subtle, mathematically targeted sound waves mixed into background noise that are imperceptible to human ears but cause an Amazon Echo or Google Home to misinterpret commands completely.
*   Research Project: Inject targeted adversarial perturbations (such as Fast Gradient Sign Method - FGSM noise) into your audio arrays. Research whether mapping these features into an exponentially large Hilbert space via quantum filters reduces an adversary's ability to corrupt the classification boundary compared to purely classical systems.